# 04 · BigQuery external table over GCS

`raw.fhvhv_trips` is an **external table**: BigQuery stores only its definition; the data stays in the
24 Parquet files in `gs://nyc-tlc-de-4821-raw/fhvhv/year=YYYY/month=MM/`. Created with
`python -m pipeline.create_raw_tables fhvhv`.

Questions:
1. Does the schema drift (`cbd_congestion_fee` only in 2025) break the table?
2. What does a query **cost**, and what drives it: columns, partitions, `LIMIT`?
3. Can we estimate the cost before running, and stop expensive queries?
4. Does BigQuery give the same answer as the local DuckDB benchmark?

**Running this notebook bills ~35 GB** of the 1 TiB monthly free tier (about 3.5%).
Every query goes through `pipeline.bq.query`, which sets a hard cap (`max_gb_billed_per_query` in `config.toml`).

In [ ]:
import pandas as pd
from google.api_core.exceptions import BadRequest

from pipeline.bq import client, dry_run_gb, query
from pipeline.config import CONFIG

TABLE = f"{CONFIG['gcp']['project_id']}.{CONFIG['bigquery']['raw_dataset']}.fhvhv_trips"
billed = []  # GB billed by each query in this notebook


def q(sql: str, **kwargs) -> pd.DataFrame:
    df = query(sql, **kwargs)
    billed.append(df.attrs["stats"].gb_billed)
    return df

## 1. Table metadata (free)

Reading a table's metadata is an API call, not a query: it bills nothing.
An external table reports 0 rows and 0 bytes: BigQuery does not know what is in the files until it reads them.
`year` and `month` are not in the files: they come from the `year=`/`month=` folder names (hive partitioning).

In [ ]:
table = client().get_table(TABLE)
print(f"type: {table.table_type} | rows: {table.num_rows} | bytes stored in BigQuery: {table.num_bytes}")
pd.DataFrame([(f.name, f.field_type) for f in table.schema], columns=["column", "type"]).T

## 2. Dry run: no estimate for external tables

A dry run validates a query and returns the bytes it would process, without running it.
For native tables that is the cost estimate shown in the console. For external tables it is always 0.

In [ ]:
dry_run_gb(f"SELECT trip_miles FROM `{TABLE}`")

## 3. Schema drift

BigQuery inferred the schema from a 2025 file, so `cbd_congestion_fee` exists in the table.
What happens for the 2024 files, which do not have the column?

In [ ]:
drift = q(f"""
    SELECT year, month,
           COUNT(*) AS trips,
           COUNTIF(cbd_congestion_fee IS NULL) AS cbd_null,
           ROUND(SUM(cbd_congestion_fee)) AS cbd_sum
    FROM `{TABLE}`
    GROUP BY year, month
    ORDER BY year, month
""")
drift

The 2024 months come back as `NULL`: no error. Good for us, but it means a missing column
and a column full of nulls look the same. Staging (dbt) should treat `cbd_congestion_fee` as 0 before 2025,
not as unknown.

**Cost check:** BigQuery bills the **logical** (uncompressed) size of the columns it reads, not the Parquet size:

| BigQuery type | Bytes per value |
|---|---|
| INT64, FLOAT64, TIMESTAMP | 8 |
| STRING | 2 + length in UTF-8 |

The query above read one FLOAT64 column for the 2025 rows only (the 2024 files have no such column).
`COUNT(*)` is free: row counts are in the Parquet metadata.

In [ ]:
rows_2025 = drift.loc[drift["year"] == 2025, "trips"].sum()
print(f"{rows_2025:,} rows x 8 bytes = {rows_2025 * 8 / 1e9:.3f} GB  vs billed {billed[-1]:.3f} GB")

## 4. What drives the cost

Three queries:
- **A.** `SELECT *` with `LIMIT 10` on one partition
- **B.** the benchmark columns, one partition, filtered by the **partition columns** `year`/`month`
- **C.** fewer columns, the same month, but filtered by `pickup_datetime` instead of the partition columns

In [ ]:
a = q(f"SELECT * FROM `{TABLE}` WHERE year = 2025 AND month = 1 LIMIT 10")

In [ ]:
b = q(f"""
    SELECT hvfhs_license_num, COUNT(*) AS trips,
           SUM(trip_miles) AS trip_miles, SUM(trip_time) AS trip_time,
           SUM(base_passenger_fare) AS base_passenger_fare, SUM(driver_pay) AS driver_pay
    FROM `{TABLE}`
    WHERE year = 2025 AND month = 1
    GROUP BY 1
""")
b

In [ ]:
c_ = q(f"""
    SELECT hvfhs_license_num, COUNT(*) AS trips, SUM(trip_miles) AS trip_miles
    FROM `{TABLE}`
    WHERE pickup_datetime >= '2025-01-01' AND pickup_datetime < '2025-02-01'
    GROUP BY 1
""")
c_

- **A** billed only the 10 MB minimum. On an external table BigQuery stops reading the files once it has
  10 rows. ⚠️ On a **native** table `LIMIT` does **not** reduce cost: the whole columns are scanned.
  Do not rely on `LIMIT` to save money in general
- **B** = rows × (8 bytes x 4 numeric columns + 8 bytes for `"HV0003"`): ~40 bytes x 20.4M rows ≈ 0.82 GB
- **C** read **fewer columns** than B and returned the same month, but billed ~14x more: BigQuery cannot know which
  files hold January pickups, so it reads `pickup_datetime` in all 24 files. Only filters on the **partition columns**
  (`year`, `month`) skip files

## 5. Safety net: `maximum_bytes_billed`

The benchmark query over all 24 months needs ~19 GB. With a 5 GB cap, BigQuery refuses it.
The dry run could not warn us (section 2), but at execution BigQuery reads the file metadata, sees the size,
and stops before reading any data.

In [ ]:
sql_all = f"""
    SELECT DATE(year, month, 1) AS month, hvfhs_license_num,
           COUNT(*) AS trips, SUM(trip_miles) AS trip_miles, SUM(trip_time) AS trip_time,
           SUM(base_passenger_fare) AS base_passenger_fare, SUM(driver_pay) AS driver_pay
    FROM `{TABLE}`
    GROUP BY ALL
    ORDER BY month, hvfhs_license_num
"""
try:
    q(sql_all, max_gb=5)
except BadRequest as e:
    print("Refused:", e.errors[0]["message"])

## 6. Same answer as DuckDB?

The same aggregation as `03_benchmark.ipynb`, now in BigQuery, compared with the DuckDB result saved by the
benchmark (`reports/benchmark/duckdb.parquet`). Partition columns replace `date_trunc(pickup_datetime)`:
same thing here, since no FHVHV trip falls outside its file's month.

In [ ]:
bq = q(sql_all)
duck = pd.read_parquet("reports/benchmark/duckdb.parquet").sort_values(["month", "hvfhs_license_num"]).reset_index(drop=True)

sums = ["trip_miles", "trip_time", "base_passenger_fare", "driver_pay"]
print("rows:", len(bq), "vs", len(duck))
print("trips equal:", (bq["trips"].values == duck["trips"].values).all())
print("max relative diff in sums:", max((abs(bq[s].astype(float).values - duck[s].values) / duck[s].values).max() for s in sums))

## 7. What this notebook cost

In [ ]:
total = sum(billed)
print(f"{len(billed)} queries billed {total:.1f} GB = {total / 1024 * 100:.1f}% of the 1 TiB monthly free tier")
print(f"Beyond the free tier this would cost about US$ {total / 1024 * 6.25:.2f}")

## 8. Findings

Measured on 2026-11-20.

- **Schema drift is handled:** files without `cbd_congestion_fee` return `NULL`. dbt staging should turn that into 0 before 2025
- **BigQuery bills logical bytes** of the columns read (8 bytes per number or timestamp, 2 + length per string),
  not the compressed Parquet size. The predictions matched the bills to the third decimal
- **Partition filters are the main lever:** filtering by `year`/`month` skips files; filtering by `pickup_datetime`
  read all 24 files (14x more for the same month)
- **Dry run says 0 for external tables**, so there is no estimate before running. `maximum_bytes_billed` still works:
  BigQuery refused the 19 GB query under a 5 GB cap and billed nothing
- **`LIMIT` was cheap here** (10 MB minimum) because BigQuery stops reading the files early. That is not true for
  native tables
- **Same results as DuckDB** for all 24 months. BigQuery took ~1.4 s of execution (~3 s with network and download)
  vs ~2.2 s for DuckDB locally. The cloud wins when the data is not on your machine, or is much larger than it

Next: dbt staging over `raw.fhvhv_trips` (step 4).